In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Project paths
DATA_DIR = Path("../data/raw/MachineLearningCSV/MachineLearningCVE")
PROCESSED_DIR = Path("../data/processed")
SAMPLES_DIR = Path("../data/samples")

# Create output directories if they don't exist
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print("Raw dataset:", DATA_DIR.resolve())
print("Processed data:", PROCESSED_DIR.resolve())
print("Samples:", SAMPLES_DIR.resolve())

Raw dataset: C:\SentinelXAI\data\raw\MachineLearningCSV\MachineLearningCVE
Processed data: C:\SentinelXAI\data\processed
Samples: C:\SentinelXAI\data\samples


In [2]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Number of CSV files: {len(csv_files)}")
print()

for file in csv_files:
    print(file.name)

Number of CSV files: 8

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [3]:
# Collect all unique labels from the raw dataset

all_labels = set()

for file in csv_files:
    for chunk in pd.read_csv(file, usecols=[" Label"], chunksize=100000):
        all_labels.update(chunk[" Label"].dropna().unique())

print("Unique labels:")
for label in sorted(all_labels):
    print(repr(label))

Unique labels:
'BENIGN'
'Bot'
'DDoS'
'DoS GoldenEye'
'DoS Hulk'
'DoS Slowhttptest'
'DoS slowloris'
'FTP-Patator'
'Heartbleed'
'Infiltration'
'PortScan'
'SSH-Patator'
'Web Attack � Brute Force'
'Web Attack � Sql Injection'
'Web Attack � XSS'


In [4]:
# Target size for the development dataset
TARGET_BENIGN = 150_000
TARGET_ATTACK = 150_000

# Random seed for reproducibility
RANDOM_STATE = 42

print("Development dataset configuration")
print("=" * 50)
print(f"Target BENIGN samples : {TARGET_BENIGN:,}")
print(f"Target ATTACK samples : {TARGET_ATTACK:,}")
print(f"Total target samples  : {TARGET_BENIGN + TARGET_ATTACK:,}")
print(f"Random state           : {RANDOM_STATE}")

Development dataset configuration
Target BENIGN samples : 150,000
Target ATTACK samples : 150,000
Total target samples  : 300,000
Random state           : 42


In [5]:
# Count available samples for each attack class

attack_counts = {}

for file in csv_files:
    for chunk in pd.read_csv(
        file,
        usecols=[" Label"],
        chunksize=100000
    ):
        attack_chunk = chunk[chunk[" Label"] != "BENIGN"]
        counts = attack_chunk[" Label"].value_counts()

        for label, count in counts.items():
            attack_counts[label] = attack_counts.get(label, 0) + count

attack_counts = pd.Series(attack_counts).sort_values(ascending=False)

print("Available attack samples")
print("=" * 60)
print(attack_counts)

Available attack samples
DoS Hulk                      231073
PortScan                      158930
DDoS                          128027
DoS GoldenEye                  10293
FTP-Patator                     7938
SSH-Patator                     5897
DoS slowloris                   5796
DoS Slowhttptest                5499
Bot                             1966
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Infiltration                      36
Web Attack � Sql Injection        21
Heartbleed                        11
dtype: int64


In [6]:
# Calculate the number of samples to take from each attack class

TARGET_ATTACK = 150_000
RANDOM_STATE = 42

# Start by keeping all rare classes with fewer than 100 samples
rare_classes = attack_counts[attack_counts < 100]

print("Rare attack classes that will be fully retained:")
print(rare_classes)

print(f"\nTotal samples from rare classes: {rare_classes.sum():,}")

Rare attack classes that will be fully retained:
Infiltration                  36
Web Attack � Sql Injection    21
Heartbleed                    11
dtype: int64

Total samples from rare classes: 68


In [7]:
# Allocate the remaining attack samples proportionally
# among the non-rare attack classes.

remaining_attack = TARGET_ATTACK - rare_classes.sum()

common_classes = attack_counts[attack_counts >= 100]

proportions = common_classes / common_classes.sum()

allocation = (proportions * remaining_attack).round().astype(int)

# Make sure the allocation sums exactly to the required number
difference = remaining_attack - allocation.sum()

if difference != 0:
    largest_class = allocation.idxmax()
    allocation[largest_class] += difference

# Add the rare classes at their full available counts
allocation = pd.concat([allocation, rare_classes])

allocation = allocation.sort_values(ascending=False)

print("Final attack sampling allocation")
print("=" * 60)
print(allocation)

print("\nTotal attack samples:", allocation.sum())

Final attack sampling allocation
DoS Hulk                      62134
PortScan                      42736
DDoS                          34426
DoS GoldenEye                  2768
FTP-Patator                    2135
SSH-Patator                    1586
DoS slowloris                  1559
DoS Slowhttptest               1479
Bot                             529
Web Attack � Brute Force        405
Web Attack � XSS                175
Infiltration                     36
Web Attack � Sql Injection       21
Heartbleed                       11
dtype: int64

Total attack samples: 150000


In [8]:
# Create the 300,000-flow development dataset
# using deterministic, class-wise sampling.

RANDOM_STATE = 42

benign_parts = []
attack_parts = []

# --------------------------------------------------
# BENIGN: collect all benign rows, then sample
# --------------------------------------------------

print("Collecting BENIGN flows...")

for file in csv_files:
    for chunk in pd.read_csv(file, chunksize=100000):
        chunk.columns = chunk.columns.str.strip()

        benign_chunk = chunk[chunk["Label"] == "BENIGN"]

        if len(benign_chunk) > 0:
            benign_parts.append(benign_chunk)

# Combine benign flows
benign_data = pd.concat(benign_parts, ignore_index=True)

print(f"Available BENIGN flows: {len(benign_data):,}")

# Sample exactly 150,000
benign_sample = benign_data.sample(
    n=TARGET_BENIGN,
    random_state=RANDOM_STATE
)

# --------------------------------------------------
# ATTACK: collect each class separately
# --------------------------------------------------

print("\nCollecting ATTACK flows...")

for label, target_count in allocation.items():

    class_parts = []

    for file in csv_files:
        for chunk in pd.read_csv(file, chunksize=100000):
            chunk.columns = chunk.columns.str.strip()

            class_chunk = chunk[chunk["Label"] == label]

            if len(class_chunk) > 0:
                class_parts.append(class_chunk)

    class_data = pd.concat(class_parts, ignore_index=True)

    # Take the required number of samples
    class_sample = class_data.sample(
        n=target_count,
        random_state=RANDOM_STATE
    )

    attack_parts.append(class_sample)

    print(
        f"{label}: "
        f"{len(class_data):,} available → "
        f"{len(class_sample):,} selected"
    )

# Combine attack samples
attack_sample = pd.concat(
    attack_parts,
    ignore_index=True
)

# --------------------------------------------------
# Combine BENIGN + ATTACK
# --------------------------------------------------

development_data = pd.concat(
    [benign_sample, attack_sample],
    ignore_index=True
)

# Shuffle the final dataset
development_data = development_data.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

# --------------------------------------------------
# Create binary target
# --------------------------------------------------

development_data["BinaryLabel"] = (
    development_data["Label"] != "BENIGN"
).astype(int)

print("\n" + "=" * 60)
print("Development dataset created!")
print("=" * 60)

print(f"Total flows: {len(development_data):,}")
print(
    f"BENIGN: "
    f"{(development_data['BinaryLabel'] == 0).sum():,}"
)
print(
    f"ATTACK: "
    f"{(development_data['BinaryLabel'] == 1).sum():,}"
)

Available BENIGN flows: 2,273,097

DoS Hulk: 231,073 available → 62,134 selected
PortScan: 158,930 available → 42,736 selected
DDoS: 128,027 available → 34,426 selected
DoS GoldenEye: 10,293 available → 2,768 selected
FTP-Patator: 7,938 available → 2,135 selected
SSH-Patator: 5,897 available → 1,586 selected
DoS slowloris: 5,796 available → 1,559 selected
DoS Slowhttptest: 5,499 available → 1,479 selected
Bot: 1,966 available → 529 selected
Web Attack � Brute Force: 1,507 available → 405 selected
Web Attack � XSS: 652 available → 175 selected
Infiltration: 36 available → 36 selected
Web Attack � Sql Injection: 21 available → 21 selected
Heartbleed: 11 available → 11 selected

Development dataset created!
Total flows: 300,000
BENIGN: 150,000
ATTACK: 150,000


In [9]:
# Verify the development dataset

print("Development Dataset Verification")
print("=" * 60)

print(f"Shape: {development_data.shape}")

print("\nColumns:")
print(f"Total columns: {len(development_data.columns)}")

print("\nBinary label distribution:")
print(development_data["BinaryLabel"].value_counts())

print("\nOriginal label distribution:")
print(development_data["Label"].value_counts())

print("\nMissing values:")
print(development_data.isna().sum().sum())

print("\nInfinite values:")
numeric_data = development_data.select_dtypes(include="number")
print(np.isinf(numeric_data).sum().sum())

print("\nDuplicate rows:")
print(development_data.duplicated().sum())

Development Dataset Verification
Shape: (300000, 80)

Columns:
Total columns: 80

Binary label distribution:
BinaryLabel
0    150000
1    150000
Name: count, dtype: int64

Original label distribution:
Label
BENIGN                        150000
DoS Hulk                       62134
PortScan                       42736
DDoS                           34426
DoS GoldenEye                   2768
FTP-Patator                     2135
SSH-Patator                     1586
DoS slowloris                   1559
DoS Slowhttptest                1479
Bot                              529
Web Attack � Brute Force         405
Web Attack � XSS                 175
Infiltration                      36
Web Attack � Sql Injection        21
Heartbleed                        11
Name: count, dtype: int64

Missing values:
283

Infinite values:
495

Duplicate rows:
28289


In [10]:
# Clean column names
development_data.columns = development_data.columns.str.strip()

# Replace infinite values with NaN
development_data = development_data.replace(
    [np.inf, -np.inf],
    np.nan
)

# Separate feature columns from target/metadata
feature_columns = [
    col for col in development_data.columns
    if col not in ["Label", "BinaryLabel"]
]

print(f"Number of feature columns: {len(feature_columns)}")

# Check missing values in features
missing_before = development_data[feature_columns].isna().sum().sum()

print(f"Missing feature values before imputation: {missing_before:,}")

Number of feature columns: 78
Missing feature values before imputation: 778


In [11]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = development_data[feature_columns].copy()
y = development_data["BinaryLabel"].copy()

# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# Second split: divide the temporary 30% into
# 15% validation and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print("Dataset split")
print("=" * 50)

print(f"Training   : {X_train.shape}")
print(f"Validation : {X_val.shape}")
print(f"Test       : {X_test.shape}")

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nValidation label distribution:")
print(y_val.value_counts())

print("\nTest label distribution:")
print(y_test.value_counts())

Dataset split
Training   : (210000, 78)
Validation : (45000, 78)
Test       : (45000, 78)

Training label distribution:
BinaryLabel
0    105000
1    105000
Name: count, dtype: int64

Validation label distribution:
BinaryLabel
1    22500
0    22500
Name: count, dtype: int64

Test label distribution:
BinaryLabel
1    22500
0    22500
Name: count, dtype: int64


In [12]:
from sklearn.impute import SimpleImputer

# Create median imputer
imputer = SimpleImputer(strategy="median")

# Fit ONLY on training data
X_train_imputed = imputer.fit_transform(X_train)

# Apply the same imputer to validation and test data
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

print("After median imputation:")
print("=" * 50)

print("Training missing values:",
      np.isnan(X_train_imputed).sum())

print("Validation missing values:",
      np.isnan(X_val_imputed).sum())

print("Test missing values:",
      np.isnan(X_test_imputed).sum())

print("\nShapes:")
print("Training   :", X_train_imputed.shape)
print("Validation :", X_val_imputed.shape)
print("Test       :", X_test_imputed.shape)

After median imputation:
Training missing values: 0
Validation missing values: 0
Test missing values: 0

Shapes:
Training   : (210000, 78)
Validation : (45000, 78)
Test       : (45000, 78)


In [13]:
from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train_scaled = scaler.fit_transform(X_train_imputed)

# Transform validation and test using the same scaler
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Feature scaling complete")
print("=" * 50)

print("Training shape   :", X_train_scaled.shape)
print("Validation shape :", X_val_scaled.shape)
print("Test shape       :", X_test_scaled.shape)

print("\nTraining mean (first 5 features):")
print(X_train_scaled[:, :5].mean(axis=0))

print("\nTraining std (first 5 features):")
print(X_train_scaled[:, :5].std(axis=0))

Feature scaling complete
Training shape   : (210000, 78)
Validation shape : (45000, 78)
Test shape       : (45000, 78)

Training mean (first 5 features):
[-3.51887831e-18 -6.11743460e-17 -5.41365894e-19 -4.06024420e-19
 -5.41365894e-19]

Training std (first 5 features):
[1. 1. 1. 1. 1.]


In [14]:
# Prepare data for LSTM
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
X_val_lstm = X_val_scaled.reshape(X_val_scaled.shape[0], X_val_scaled.shape[1], 1)
X_test_lstm = X_test_scaled.reshape(X_test_scaled.shape[1] if False else X_test_scaled.shape[0],
                                     X_test_scaled.shape[1], 1)

print("LSTM data preparation complete")
print("=" * 50)
print("Training shape   :", X_train_lstm.shape)
print("Validation shape :", X_val_lstm.shape)
print("Test shape       :", X_test_lstm.shape)

LSTM data preparation complete
Training shape   : (210000, 78, 1)
Validation shape : (45000, 78, 1)
Test shape       : (45000, 78, 1)


In [15]:
# Prepare target labels for the IDS
y_train_lstm = y_train.to_numpy()
y_val_lstm = y_val.to_numpy()
y_test_lstm = y_test.to_numpy()

print("Target labels prepared")
print("=" * 50)

print("Training labels   :", y_train_lstm.shape)
print("Validation labels :", y_val_lstm.shape)
print("Test labels       :", y_test_lstm.shape)

print("\nTraining label distribution:")
print(pd.Series(y_train_lstm).value_counts().sort_index())

print("\nValidation label distribution:")
print(pd.Series(y_val_lstm).value_counts().sort_index())

print("\nTest label distribution:")
print(pd.Series(y_test_lstm).value_counts().sort_index())

Target labels prepared
Training labels   : (210000,)
Validation labels : (45000,)
Test labels       : (45000,)

Training label distribution:
0    105000
1    105000
Name: count, dtype: int64

Validation label distribution:
0    22500
1    22500
Name: count, dtype: int64

Test label distribution:
0    22500
1    22500
Name: count, dtype: int64


In [16]:
# Save preprocessed data for the baseline IDS notebook

np.save(PROCESSED_DIR / "X_train_lstm.npy", X_train_lstm)
np.save(PROCESSED_DIR / "X_val_lstm.npy", X_val_lstm)
np.save(PROCESSED_DIR / "X_test_lstm.npy", X_test_lstm)

np.save(PROCESSED_DIR / "y_train_lstm.npy", y_train_lstm)
np.save(PROCESSED_DIR / "y_val_lstm.npy", y_val_lstm)
np.save(PROCESSED_DIR / "y_test_lstm.npy", y_test_lstm)

print("Preprocessed LSTM data saved successfully.")

Preprocessed LSTM data saved successfully.
